# Week 82, Day 1: Distributed systems concepts

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w082d1.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
events = [
    ("server-a", 1000.000, "ticket 7 created"),
    ("server-b", 999.998, "ticket 7 assigned"),     # server-b's clock runs 5 ms slow
    ("server-a", 1000.010, "ticket 7 resolved"),
]
for server, t, what in sorted(events, key=lambda e: e[1]):
    print(f"{t:.3f} {server}: {what}")

In [ ]:
def compare(a, b):
    nodes = set(a) | set(b)
    a_le_b = all(a.get(n, 0) <= b.get(n, 0) for n in nodes)
    b_le_a = all(b.get(n, 0) <= a.get(n, 0) for n in nodes)
    if a_le_b and b_le_a:
        return "equal"
    if a_le_b:
        return "before"
    if b_le_a:
        return "after"
    return "concurrent"

edit_on_a = {"a": 2, "b": 1}
edit_on_b = {"a": 1, "b": 2}
later = {"a": 2, "b": 2}
print(compare(edit_on_a, edit_on_b), compare(edit_on_a, later), compare(later, edit_on_b))

> **Key idea.** Distributed systems fail partially; a timeout can't tell lost, slow, crashed-before and crashed-after apart, so requests that time out may have succeeded (make them idempotent). Heartbeats with a threshold suspect failures, trading speed for false alarms. Clocks drift and jump: use monotonic clocks for durations and logical clocks for order. Lamport clocks (max(own, received) + 1) respect happened-before; vector clocks (element-wise max, then increment your own entry) detect concurrent events. Raft elects one leader per term by majority vote of all nodes (one vote per node per term, only for up-to-date candidates); higher terms make stale leaders step down; odd cluster sizes. Two-phase commit (prepare, then commit only if all vote yes) is atomic but can block; sagas run local steps and, on failure, run completed steps' compensations in reverse.

### Exercise: Lamport clocks

Write a class `LamportClock(node)` with `tick()` (a local event: increment and return the time), `send()` (increment and return `(time, node)`, the stamp to attach to a message) and `receive(stamp)` (set the time to `max(own, stamp_time) + 1` and return it). Then `total_order(events)`: `events` is a list of `(time, node, description)`; return the descriptions sorted by time, ties broken by node name.

In [ ]:
class LamportClock:
    ___

def total_order(events):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_lamport():\n    \"happened-before is respected\"\n    a, b = LamportClock(\"a\"), LamportClock(\"b\")\n    created = a.tick()\n    stamp = a.send()\n    assert (created, stamp) == (1, (2, \"a\"))\n    b.tick()\n    assigned = b.receive(stamp)\n    assert assigned == 3\n    for _ in range(5):\n        b.tick()\n    resolved_stamp = b.send()\n    assert a.receive(resolved_stamp) == 10\n    events = [(created, \"a\", \"created\"), (assigned, \"b\", \"assigned\"), (3, \"a\", \"noted\"), (1, \"b\", \"booted\")]\n    assert total_order(events) == [\"created\", \"booted\", \"noted\", \"assigned\"]")

### Exercise: Vector clocks

Write a class `VectorClock(node)` holding a dict of counters (missing nodes count as 0). `event()` increments its own entry and returns a **copy** of the vector; `send()` is the same as `event()`; `receive(vector)` takes the element-wise maximum with the received vector, then increments its own entry, and returns a copy. Then `compare(a, b)` for two vectors: `"equal"`, `"before"` (a happened before b), `"after"` or `"concurrent"`. And `conflicts(versions)`: given a list of `(vector, value)` versions of one item, return the values of the versions that **no other version happened after** (the ones still in play), in their original order.

In [ ]:
class VectorClock:
    ___

def compare(a, b):
    ___

def conflicts(versions):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_vectors():\n    \"before, after and concurrent\"\n    a, b = VectorClock(\"a\"), VectorClock(\"b\")\n    v1 = a.event()\n    msg = a.send()\n    v2 = b.receive(msg)\n    v3 = a.event()\n    assert v1 == {\"a\": 1} and msg == {\"a\": 2} and v2 == {\"a\": 2, \"b\": 1} and v3 == {\"a\": 3}\n    assert compare(v1, v2) == \"before\" and compare(v2, v1) == \"after\" and compare(v2, v3) == \"concurrent\"\n    assert compare({\"a\": 1}, {\"a\": 1, \"b\": 0}) == \"equal\"\n    v1[\"a\"] = 99\n    assert a.clock[\"a\"] == 3, \"return copies\"\n\ndef test_conflicts():\n    \"siblings survive\"\n    versions = [({\"a\": 1}, \"draft\"), ({\"a\": 2}, \"Ada's edit\"), ({\"a\": 1, \"b\": 1}, \"Bob's edit\"), ({\"a\": 2}, \"Ada's edit again\")]\n    assert conflicts(versions) == [\"Ada's edit\", \"Bob's edit\", \"Ada's edit again\"]\n    assert conflicts([({\"a\": 1}, \"x\"), ({\"a\": 2, \"b\": 1}, \"y\")]) == [\"y\"]")

### Exercise: A heartbeat failure detector

Write a class `FailureDetector(clock, timeout)`. `heartbeat(node)` records that the node was heard from now. `suspected()` returns the sorted nodes not heard from for **more than** `timeout` seconds. `status()` returns a dict from each node to `"alive"` or `"suspected"`, sorted by node. And `false_alarms(silences)`: given a list of `(node, gap_seconds)` gaps between one node's heartbeats in normal running, the number of gaps longer than the timeout (each would have been a false suspicion).

In [ ]:
class FailureDetector:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_detector():\n    \"silent too long means suspected\"\n    class Clock:\n        t = 0\n        def __call__(self):\n            return self.t\n    clock = Clock()\n    d = FailureDetector(clock, timeout=5)\n    for n in (\"a\", \"b\", \"c\"):\n        d.heartbeat(n)\n    clock.t = 4\n    d.heartbeat(\"a\")\n    clock.t = 5\n    assert d.suspected() == []\n    clock.t = 5.5\n    assert d.suspected() == [\"b\", \"c\"] and d.status() == {\"a\": \"alive\", \"b\": \"suspected\", \"c\": \"suspected\"}\n    d.heartbeat(\"b\")\n    assert d.suspected() == [\"c\"]\n    assert d.false_alarms([(\"a\", 1), (\"a\", 6), (\"b\", 5), (\"b\", 9.5)]) == 2")

### Exercise: Votes and terms

Write a class `Node(name, cluster_size, last_log_term=0, last_log_index=0)` for Raft-style elections, with `term` (starting at 0), `voted_for` (`None`) and `role` (`"follower"`). `request_vote(term, candidate, cand_log_term, cand_log_index)` returns `(self.term, granted)`: a request with a **lower** term than the node's is refused; a **higher** term makes the node adopt that term, forget its vote and become a follower first. Then grant the vote only if the node hasn't voted for someone else this term **and** the candidate's log is at least as up to date (its last log term is higher, or equal with an index at least as high); record the vote. Then `run_election(candidate, others)`: the candidate becomes `"candidate"`, increments its term, votes for itself, asks each of the other nodes (a list of `Node`s) with its own log position, and becomes `"leader"` if its votes (including its own) are **more than half of `cluster_size`**; otherwise it stays a candidate. Return the number of votes.

In [ ]:
class Node:
    ___

def run_election(candidate, others):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_votes():\n    \"one vote per term, only for up-to-date logs\"\n    n = Node(\"n1\", 5, last_log_term=2, last_log_index=7)\n    assert n.request_vote(1, \"x\", 9, 9) == (1, True)\n    assert n.request_vote(1, \"y\", 9, 9) == (1, False) and n.request_vote(1, \"x\", 9, 9) == (1, True)\n    assert n.request_vote(0, \"z\", 9, 9) == (1, False)\n    assert n.request_vote(2, \"stale\", 2, 6) == (2, False) and n.request_vote(2, \"fresh\", 2, 7) == (2, True)\n    assert n.request_vote(3, \"newer\", 3, 1) == (3, True)\n\ndef test_elections():\n    \"a majority of the whole cluster\"\n    nodes = [Node(f\"n{i}\", 5, 1, 5) for i in range(5)]\n    assert run_election(nodes[0], nodes[1:3]) == 3 and nodes[0].role == \"leader\" and nodes[0].term == 1\n    lagging = Node(\"lag\", 5, 1, 2)\n    others = [Node(f\"m{i}\", 5, 1, 5) for i in range(4)]\n    assert run_election(lagging, others) == 1 and lagging.role == \"candidate\"\n    four = [Node(f\"f{i}\", 4) for i in range(4)]\n    assert run_election(four[0], four[1:2]) == 2 and four[0].role == \"candidate\"\n    leader = nodes[0]\n    leader.request_vote(7, \"someone\", 9, 9)\n    assert leader.role == \"follower\" and leader.term == 7")

### Exercise: Two-phase commit

Write `two_phase_commit(participants)`: each participant is a dict with `name`, `prepare` (a function returning `True` for "yes", `False` for "no", or raising `TimeoutError` for no answer) and lists `log` to which you append what happened to it. Phase 1: ask each participant to prepare, in order, stopping at the first that doesn't vote yes; append `"prepared"` to each that voted yes. Phase 2: if **all** voted yes, append `"committed"` to every participant and return `("committed", [])`; otherwise append `"aborted"` to every participant that had prepared and return `("aborted", [NAMES that voted no or timed out])`.

In [ ]:
def two_phase_commit(participants):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def _p(name, answer):\n    def prepare():\n        if answer == \"timeout\":\n            raise TimeoutError\n        return answer\n    return {\"name\": name, \"prepare\": prepare, \"log\": []}\n\ndef test_commit():\n    \"all yes\"\n    ps = [_p(\"payments\", True), _p(\"seats\", True), _p(\"email\", True)]\n    assert two_phase_commit(ps) == (\"committed\", [])\n    assert all(p[\"log\"] == [\"prepared\", \"committed\"] for p in ps)\n\ndef test_abort():\n    \"any no, or silence, aborts everyone who prepared\"\n    ps = [_p(\"payments\", True), _p(\"seats\", False), _p(\"email\", True)]\n    assert two_phase_commit(ps) == (\"aborted\", [\"seats\"])\n    assert [p[\"log\"] for p in ps] == [[\"prepared\", \"aborted\"], [], []]\n    ps = [_p(\"payments\", True), _p(\"seats\", True), _p(\"email\", \"timeout\")]\n    assert two_phase_commit(ps) == (\"aborted\", [\"email\"])\n    assert [p[\"log\"] for p in ps] == [[\"prepared\", \"aborted\"], [\"prepared\", \"aborted\"], []]")

### Exercise: A saga

Write `run_saga(steps)`: each step is `(name, action, compensation)`, where both are functions with no arguments. Run the actions in order. If one raises, run the compensations of the steps that **completed**, in **reverse** order, then return `("compensated", FAILED_STEP_NAME, [names of compensated steps in the order compensated])`. If all succeed, return `("done", None, [])`. If a compensation itself raises, carry on with the others, but put `NAME (compensation failed)` in the list instead of `NAME`, so a person knows to fix that one by hand.

In [ ]:
def run_saga(steps):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_saga():\n    \"undo in reverse\"\n    log = []\n    def act(name, fail=False):\n        def f():\n            if fail:\n                raise RuntimeError(name)\n            log.append(name)\n        return f\n    def undo(name, fail=False):\n        def f():\n            if fail:\n                raise RuntimeError(\"undo \" + name)\n            log.append(\"undo \" + name)\n        return f\n    ok = [(\"charge\", act(\"charge\"), undo(\"charge\")), (\"seat\", act(\"seat\"), undo(\"seat\"))]\n    assert run_saga(ok) == (\"done\", None, []) and log == [\"charge\", \"seat\"]\n    log.clear()\n    bad = [(\"charge\", act(\"charge\"), undo(\"charge\")), (\"seat\", act(\"seat\"), undo(\"seat\")), (\"ticket\", act(\"ticket\", True), undo(\"ticket\"))]\n    assert run_saga(bad) == (\"compensated\", \"ticket\", [\"seat\", \"charge\"]) and log == [\"charge\", \"seat\", \"undo seat\", \"undo charge\"]\n    log.clear()\n    worse = [(\"charge\", act(\"charge\"), undo(\"charge\", True)), (\"seat\", act(\"seat\"), undo(\"seat\")), (\"ticket\", act(\"ticket\", True), undo(\"ticket\"))]\n    assert run_saga(worse) == (\"compensated\", \"ticket\", [\"seat\", \"charge (compensation failed)\"])")

### Bug hunt: Ordering by wall clocks

Events from two servers are ordered by their wall-clock timestamps, and server-b's clock runs slow, so a reply appears before the message it answers: this prints `['reply', 'question']`. Order them by Lamport timestamps instead (each event's `lamport` field), so it prints `['question', 'reply']`.

In [ ]:
events = [
    {"what": "question", "wall": 1000.004, "lamport": 4},
    {"what": "reply", "wall": 1000.001, "lamport": 5},     # sent after receiving the question
]
print([e["what"] for e in sorted(events, key=lambda e: e["wall"])])

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"logical time\"\n    expect_output(\"['question', 'reply']\")")

### Bug hunt: Two leaders

The majority check uses `>=` with half the cluster, so in a four-node cluster split in two, **both** halves think they've won: this prints `side A leads: True, side B leads: True`. A majority is **more than** half of all nodes, so it prints `side A leads: False, side B leads: False`.

In [ ]:
cluster_size = 4

def wins(votes):
    return votes >= cluster_size / 2

print(f"side A leads: {wins(2)}, side B leads: {wins(2)}")

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"strictly more than half\"\n    expect_output(\"side A leads: False, side B leads: False\")")

### Bug hunt: Undoing in the wrong order

The saga compensates the completed steps in the order they ran, so it refunds the card before cancelling the hotel booking that the refund depends on: this prints `['refund card', 'cancel hotel']`. Compensate in reverse order, so it prints `['cancel hotel', 'refund card']`.

In [ ]:
completed = [("charge card", "refund card"), ("book hotel", "cancel hotel")]
undo = [comp for _, comp in completed]
print(undo)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"last done, first undone\"\n    expect_output(\"['cancel hotel', 'refund card']\")")

### Puzzle: Receiving with a vector clock

Arrange the receive operation of a vector clock. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
p = ___   # a Lamport clock at 4 receives a stamp of 9: its new time
q = ___   # how many failures a 5-node Raft cluster survives
r = ___   # compare({"a": 1, "b": 0}, {"a": 0, "b": 1}): before, after, equal or concurrent
s = ___   # the clock to use for measuring a duration on one machine ("wall" or "monotonic")
t = ___   # True or False: a request that timed out definitely failed

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == 10 and q == 2 and r == \"concurrent\", \"p, q and r\"\n    assert s == \"monotonic\" and t is False, \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: replicating a log, Raft style

Raft's leader replicates its log with **AppendEntries** messages. Each log entry is `(term, command)`; indexes count from 1. Write a class `Follower` with `log` (a list) and `append_entries(prev_index, prev_term, entries, leader_commit)`: if `prev_index > 0` and the follower has no entry at `prev_index` or its term there isn't `prev_term`, reply `False` (the consistency check). Otherwise, for each new entry at position `prev_index + 1, ...`: if the follower already has an entry there with a **different** term, delete it and everything after it; append any entries it doesn't have. Then set `commit_index` to `min(leader_commit, len(log))` if that's higher than before, and reply `True`. Then a class `Leader(term, log, followers)` with `replicate()`: for each follower, starting from `next_index[f] = len(leader log) + 1` initially, send the entries from `next_index` on with the previous entry's index and term; on `False`, decrease that follower's `next_index` by one and retry until it succeeds (at worst from index 1); on success set `match_index[f]` to the last index sent. Then advance `commit_index` to the highest index `N` such that a **majority** of the cluster (the leader plus followers) has the entry (`match_index >= N`, counting the leader itself) **and** the entry at `N` is from the leader's current term. `replicate` sends `leader_commit` with each message and returns the new commit index.

In [ ]:
class Follower:
    ___

class Leader:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_follower():\n    \"the consistency check\"\n    f = Follower()\n    assert f.append_entries(0, 0, [(1, \"a\"), (1, \"b\")], 1) is True and f.log == [(1, \"a\"), (1, \"b\")] and f.commit_index == 1\n    assert f.append_entries(3, 1, [(1, \"d\")], 1) is False\n    assert f.append_entries(2, 2, [(2, \"c\")], 1) is False\n    assert f.append_entries(2, 1, [(2, \"c\")], 3) is True and f.log == [(1, \"a\"), (1, \"b\"), (2, \"c\")] and f.commit_index == 3\n    assert f.append_entries(1, 1, [(3, \"x\")], 2) is True and f.log == [(1, \"a\"), (3, \"x\")] and f.commit_index == 3\n    assert f.append_entries(1, 1, [(3, \"x\")], 1) is True and f.log == [(1, \"a\"), (3, \"x\")]\n\ndef test_leader():\n    \"catching followers up and committing on a majority\"\n    up_to_date, behind, divergent = Follower(), Follower(), Follower()\n    base = [(1, \"set a\"), (1, \"set b\"), (2, \"set c\")]\n    up_to_date.log = list(base)\n    behind.log = base[:1]\n    divergent.log = [(1, \"set a\"), (1, \"set b\"), (1, \"stale write\"), (1, \"stale write 2\")]\n    leader = Leader(term=3, log=base + [(3, \"set d\")], followers=[up_to_date, behind, divergent, Follower()])\n    assert leader.replicate() == 4\n    for f in (up_to_date, behind, divergent):\n        assert f.log == leader.log\n    assert leader.followers[3].log == leader.log and leader.match_index[behind] == 4\n\ndef test_old_terms_not_committed_alone():\n    \"only entries from the current term are committed by counting\"\n    a, b = Follower(), Follower()\n    leader = Leader(term=4, log=[(2, \"old\")], followers=[a, b])\n    assert leader.replicate() == 0 and a.log == [(2, \"old\")]\n    leader.log.append((4, \"new\"))\n    assert leader.replicate() == 2 and a.commit_index == 0\n    assert leader.replicate() == 2 and a.commit_index == 2")

## Warm-up

Sorting by each server's clock puts "assigned" **before** "created", which is impossible. Server clocks drift and are corrected in jumps, so timestamps from different machines can't be trusted to order events closer together than the drift. Distributed systems order events with **logical clocks** instead.

## Partial failure

On one computer, a program usually works or crashes as a whole. In a distributed system, **parts** fail while others carry on: one server crashes, one network link drops messages, one disk slows down. And from the outside, many failures look the same: you sent a request and no answer came. Was the request lost? Did the server crash before doing the work, or **after** doing it but before replying? Is it just slow, or overloaded, or paused for garbage collection?

**A timeout can't tell these apart.** All it says is "no answer yet". So:

- choose timeouts deliberately (from the p99 latency, Week 81, plus a margin), and remember a timed-out request **may have succeeded** (Week 77's idempotency keys exist for exactly this);
- detect failed machines with **heartbeats**: each node sends "I'm alive" regularly; one silent for longer than a threshold is **suspected** dead. It might just be slow, so the threshold trades speed of detection against false alarms;
- design so that a node wrongly declared dead can't do harm when it comes back (Raft's **terms**, below, are one way).

The famous **two generals problem** shows the limit: two parties communicating over a link that can lose messages can never be **certain** they agree, however many acknowledgements they exchange, because the last message can always be lost. Real systems settle for "agree with high probability, and recover when we don't".

## Time and order

Clocks on different machines disagree by milliseconds (or seconds), and NTP corrections make a clock jump, even backwards. So:

- for **measuring durations** on one machine, use a **monotonic** clock (`time.monotonic()`), which never jumps backwards (Week 78);
- for **ordering events across machines**, use **logical clocks**.

A **Lamport clock** is a counter on each node: increment it before each local event or send; attach it to each message; on receiving, set the counter to `max(own, received) + 1`. Then if event A **happened before** B (A came first on the same node, or A was a send and B its receive, or a chain of these), A's timestamp is lower. Ties are broken by node id, giving a **total order** everyone agrees on. But the converse doesn't hold: a lower timestamp doesn't prove A happened first; they may be unrelated.

A **vector clock** fixes that: each node keeps a counter **per node**. A node increments its own entry on each event; on receiving, it takes the element-wise maximum of the two vectors, then increments its own. Comparing two vectors tells you the relationship exactly: A happened before B if every entry of A is at most B's and at least one is smaller; if neither is before the other, the events are **concurrent**, meaning neither could know about the other. Databases such as DynamoDB's predecessors used them to spot conflicting writes (two people editing the same item at once) instead of silently losing one.

## Agreement

Many systems need nodes to **agree**: on which node is the leader, or on the order of a log of changes (Week 81's replication). Doing that safely, despite crashes and lost messages, is **consensus**, and the algorithm most systems use today is **Raft** (etcd, Consul, CockroachDB, and the controller in Kafka).

Raft's core ideas, simplified:

- Time is divided into numbered **terms**. Each term has at most one leader.
- A follower that hears nothing from a leader for a while becomes a **candidate**: it increases the term, votes for itself and asks the others for votes.
- A node grants **at most one vote per term**, and only to a candidate whose log is at least as up to date as its own (so a stale node can't become leader and erase committed entries).
- A candidate with votes from a **majority** (more than half of **all** nodes, not just those that answered) becomes leader. Two majorities always share a node, so two leaders can't be elected in the same term.
- Any node that sees a message with a **higher term** steps down to follower. So a leader that was cut off and comes back finds a higher term and stops acting as leader.

With five nodes, a majority is three, so the cluster survives two failures; with four, a majority is still three, so it survives only one. That's why clusters have **odd** sizes.

## Transactions across services

A transaction in one database is atomic (Week 75). When one operation spans **several services** (charge the card, reserve the seat, send the ticket), there are two main approaches:

- **Two-phase commit** (2PC): a **coordinator** asks every participant to **prepare** (do everything except make it permanent, and promise to commit if asked). If **all** vote yes, it tells them to **commit**; if any votes no (or doesn't answer), it tells them all to **abort**. It's atomic, but participants hold locks while waiting, and if the coordinator dies between the phases, they're stuck. It's used inside databases more than between services.
- **Sagas**: run the steps one after another, each a normal local transaction, and give each step a **compensation** that undoes it (refund the card, release the seat). If a step fails, run the compensations of the steps that **already completed**, in **reverse** order. Nothing is locked for long, at the cost of a moment when the system is visibly half-done, and compensations that must be written carefully (and be idempotent, because they may be retried).

## Practice

## Quiz

## Daily challenge

## Before you go

Look up the Raft visualisation at raft.github.io and watch an election and a log being replicated; you've now written the core of both. Tomorrow: **observability**, how to know what a distributed system is actually doing.